# Day 4B: BERT, RoBERTa, and Fine-Tuning for Text Classification

This is the advanced transformer companion to Day 4. It shows how BERT-style models are used for classification and how fine-tuning differs from classical feature engineering.

The notebook is safe to open without internet access: cells that download model weights are disabled by default. To run the transformer parts, install the optional transformer requirements and set the run flags to `True`.

By the end you should be able to:

1. Explain how transformer tokenization differs from spaCy/scikit-learn tokenization.
2. Run a pretrained transformer classifier and discuss domain mismatch.
3. Fine-tune BERT/RoBERTa-style sequence classifiers on local SMS spam data.
4. Compare transformer models to a TF-IDF logistic-regression baseline.
5. Report transformer models with enough detail for reproducibility.

In [ ]:
from pathlib import Path
import importlib.util
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, precision_recall_fscore_support, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

pd.set_option('display.max_colwidth', 160)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Setup flags

The first run of a Hugging Face model will download weights. For a live class, do this before the session if possible.

Recommended class setting:

- Use `distilbert-base-uncased` for a faster CPU/GPU demonstration.
- Show that switching to `bert-base-uncased` or `roberta-base` changes the model/tokenizer but not the training logic.
- Keep sample sizes small for live fine-tuning.

In [ ]:
RUN_TRANSFORMER_TOKENIZATION = False
RUN_PRETRAINED_PIPELINE = False
RUN_FROZEN_FEATURES = False
RUN_FINE_TUNING = False

# Fast classroom default. Replace with 'bert-base-uncased' or 'roberta-base' to demonstrate those architectures.
MODEL_NAME = 'distilbert-base-uncased'
ALTERNATIVE_MODELS = ['bert-base-uncased', 'roberta-base']

TRANSFORMERS_AVAILABLE = importlib.util.find_spec('transformers') is not None
TORCH_AVAILABLE = importlib.util.find_spec('torch') is not None

print('transformers available:', TRANSFORMERS_AVAILABLE)
print('torch available:', TORCH_AVAILABLE)
print('model selected:', MODEL_NAME)

In [ ]:
def find_data_dir():
    candidates = [Path('../data'), Path('materials/data'), Path('data')]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError('Could not find the course data directory.')

DATA_DIR = find_data_dir()
DATA_DIR

## 1. Load and split the SMS spam data

We use the local SMS Spam Collection because it is short, labeled, imbalanced, and easy to interpret. For live fine-tuning, we deliberately sample a small balanced subset so the demo can run in class.

In [ ]:
sms = pd.read_csv(DATA_DIR / 'sms_spam.csv')
sms = sms[['text', 'label', 'is_spam']].dropna().copy()
sms = sms.rename(columns={'label': 'class_name', 'is_spam': 'label'})
sms['label'] = sms['label'].astype(int)
sms['class_name'] = sms['class_name'].str.lower()


def sample_per_class(frame, class_col, n, seed):
    pieces = []
    for _, group in frame.groupby(class_col):
        pieces.append(group.sample(min(len(group), n), random_state=seed))
    return pd.concat(pieces, ignore_index=True)

small = (
    sample_per_class(sms, 'class_name', 600, RANDOM_SEED)
    .sample(frac=1, random_state=RANDOM_SEED)
    .reset_index(drop=True)
)

train_df, temp_df = train_test_split(
    small,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=small['label']
)
valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_df['label']
)

print(train_df.shape, valid_df.shape, test_df.shape)
train_df[['text', 'class_name', 'label']].head()

## 2. Classical baseline: TF-IDF + logistic regression

Always keep a simple baseline. If BERT barely beats this, that is a substantive result, not a failure.

In [ ]:
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', min_df=2, max_features=8000, ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

baseline.fit(train_df['text'], train_df['class_name'])
baseline_pred = baseline.predict(test_df['text'])

print(classification_report(test_df['class_name'], baseline_pred))
ConfusionMatrixDisplay.from_predictions(test_df['class_name'], baseline_pred, cmap='Blues')
plt.title('TF-IDF logistic-regression baseline')
plt.tight_layout()

## 3. Transformer tokenization

BERT-style models do not use spaCy tokens. They use model-specific subword tokenizers. BERT uses WordPiece; RoBERTa uses a byte-level BPE tokenizer. This matters because the unit of representation changes.

### Methodology formulas: transformer classification and fine-tuning

A transformer tokenizer maps text to a sequence of token ids, usually with special tokens:

$$x_i \rightarrow ([\mathrm{CLS}], t_1, \ldots, t_m, [\mathrm{SEP}]).$$

Self-attention builds contextual representations by comparing queries, keys, and values:

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V.$$

For sequence classification, BERT-style models usually classify from the contextual representation of the special classification token:

$$p(y_i \mid x_i)=\mathrm{softmax}(W h_{i,[\mathrm{CLS}]} + b).$$

Fine-tuning minimizes cross-entropy on labeled examples:

$$\mathcal{L}(\Theta)=-\frac{1}{N}\sum_{i=1}^N \log p_\Theta(y_i \mid x_i).$$

The frozen-feature workflow keeps transformer parameters fixed and trains a small classifier on pooled embeddings, for example

$$\bar{h}_i = \frac{1}{m_i}\sum_{t=1}^{m_i} h_{it}, \qquad P(y_i=1\mid \bar{h}_i)=\sigma(\eta_0+\bar{h}_i^\top\eta).$$

The full fine-tuning workflow updates both the classifier head and the transformer weights:

$$\Theta_{r+1}=\Theta_r - \lambda \nabla_\Theta \mathcal{L}(\Theta_r).$$


In [ ]:
if RUN_TRANSFORMER_TOKENIZATION:
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError('Install transformers first: pip install -r materials/requirements-transformers.txt')

    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    example = test_df.iloc[0]['text']
    encoded = tokenizer(example, truncation=True, max_length=64)

    token_table = pd.DataFrame({
        'token_id': encoded['input_ids'],
        'token': tokenizer.convert_ids_to_tokens(encoded['input_ids']),
        'attention_mask': encoded['attention_mask']
    })
    display(example)
    display(token_table.head(40))
else:
    print('Tokenization demo skipped. Set RUN_TRANSFORMER_TOKENIZATION=True after installing transformers and model weights.')

### Additional demo: spaCy vs. BERT/RoBERTa tokenization

Transformer models use subword tokenization, so the tokens seen by the model are not the same as linguistic words from spaCy. This comparison is useful before interpreting transformer inputs.

In [ ]:
if RUN_TRANSFORMER_TOKENIZATION:
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError('Install transformers first: pip install -r materials/requirements-transformers.txt')

    import spacy
    from transformers import AutoTokenizer

    compare_text = "URGENT! You won a free prize. Reply STOP to opt out."
    spacy_tokenizer = spacy.blank('en')

    rows = []
    for position, token in enumerate(spacy_tokenizer(compare_text)):
        rows.append({'tokenizer': 'spaCy words', 'position': position, 'token': token.text})

    transformer_names = []
    for model_name in [MODEL_NAME] + ALTERNATIVE_MODELS:
        if model_name not in transformer_names:
            transformer_names.append(model_name)

    for model_name in transformer_names:
        try:
            tok = AutoTokenizer.from_pretrained(model_name)
            pieces = tok.tokenize(compare_text)
            for position, token in enumerate(pieces):
                rows.append({'tokenizer': model_name, 'position': position, 'token': token})
        except Exception as exc:
            rows.append({'tokenizer': model_name, 'position': 0, 'token': f'Could not load tokenizer: {type(exc).__name__}'})

    tokenization_comparison = pd.DataFrame(rows)
    display(tokenization_comparison)

    token_counts = tokenization_comparison.groupby('tokenizer').size().reset_index(name='n_tokens')
    plt.figure(figsize=(7, 3.5))
    sns.barplot(data=token_counts, x='n_tokens', y='tokenizer', color='#4C72B0')
    plt.title('Same sentence, different token counts')
    plt.xlabel('Number of tokens/subwords')
    plt.ylabel('')
    plt.tight_layout()
else:
    print('Tokenization comparison skipped. Set RUN_TRANSFORMER_TOKENIZATION=True to run it.')

## 4. Pretrained sentiment classifier as a domain check

This section uses a model that was already fine-tuned on sentiment data and evaluates it on the local TweetEval sentiment sample. It is useful for explaining the difference between pretraining, task fine-tuning, and domain fit. It is not a spam detector.

In [ ]:
if RUN_PRETRAINED_PIPELINE:
    if not TRANSFORMERS_AVAILABLE:
        raise ImportError('Install transformers first: pip install -r materials/requirements-transformers.txt')

    from transformers import pipeline

    sentiment_model = pipeline(
        'sentiment-analysis',
        model='distilbert-base-uncased-finetuned-sst-2-english',
        truncation=True,
        device=-1
    )

    tweet_eval = pd.read_csv(DATA_DIR / 'tweet_eval_sentiment_sample.csv')
    pipe_sample = (
        tweet_eval[tweet_eval['label_name'].isin(['negative', 'positive'])]
        .sample(120, random_state=RANDOM_SEED)
        .copy()
    )
    outputs = sentiment_model(pipe_sample['text'].tolist(), batch_size=16)
    pipe_sample['transformer_label'] = [o['label'].lower() for o in outputs]
    pipe_sample['transformer_polarity'] = pipe_sample['transformer_label'].map({'positive': 'positive', 'negative': 'negative'})
    pipe_sample['transformer_score'] = [o['score'] for o in outputs]

    print(classification_report(pipe_sample['label_name'], pipe_sample['transformer_polarity']))
    ConfusionMatrixDisplay.from_predictions(pipe_sample['label_name'], pipe_sample['transformer_polarity'], cmap='Blues')
    plt.title('Pretrained DistilBERT sentiment pipeline on TweetEval')
    plt.tight_layout()
    display(pipe_sample[['text', 'label_name', 'transformer_polarity', 'transformer_score']].head())
else:
    print('Pretrained pipeline skipped. Set RUN_PRETRAINED_PIPELINE=True to run it.')

## 5. Frozen transformer features

A middle ground between feature engineering and fine-tuning is to use a transformer as a frozen feature extractor, then train a simple classifier on the resulting vectors. This is slower than TF-IDF but cheaper than full fine-tuning.

In [ ]:
if RUN_FROZEN_FEATURES:
    if not (TRANSFORMERS_AVAILABLE and TORCH_AVAILABLE):
        raise ImportError('Install transformers and torch first.')

    import torch
    from transformers import AutoModel, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME)
    encoder.eval()

    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def encode_texts(texts, batch_size=16, max_length=128):
        vectors = []
        with torch.no_grad():
            for start in range(0, len(texts), batch_size):
                batch = texts[start:start + batch_size]
                enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors='pt')
                out = encoder(**enc)
                pooled = mean_pool(out.last_hidden_state, enc['attention_mask'])
                vectors.append(pooled.cpu().numpy())
        return np.vstack(vectors)

    frozen_train = train_df.sample(300, random_state=RANDOM_SEED).copy()
    frozen_test = test_df.sample(150, random_state=RANDOM_SEED).copy()

    X_train = encode_texts(frozen_train['text'].tolist())
    X_test = encode_texts(frozen_test['text'].tolist())

    frozen_clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    frozen_clf.fit(X_train, frozen_train['class_name'])
    frozen_pred = frozen_clf.predict(X_test)

    print(classification_report(frozen_test['class_name'], frozen_pred))
    ConfusionMatrixDisplay.from_predictions(frozen_test['class_name'], frozen_pred, cmap='Blues')
    plt.title(f'Frozen {MODEL_NAME} embeddings + logistic regression')
    plt.tight_layout()
else:
    print('Frozen feature extraction skipped. Set RUN_FROZEN_FEATURES=True to run it.')

## 6. Fine-tuning BERT/RoBERTa for sequence classification

Fine-tuning updates the transformer weights for the target task. This is the core distinction from frozen features.

This cell uses Hugging Face `Trainer`. It intentionally uses a small balanced SMS sample, short sequences, and one epoch for classroom speed. For a real project, increase the data, tune hyperparameters, and report uncertainty and out-of-domain validation.

In [ ]:
if RUN_FINE_TUNING:
    if not (TRANSFORMERS_AVAILABLE and TORCH_AVAILABLE):
        raise ImportError('Install the optional transformer stack first.')

    import inspect
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print('device:', device)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    ft_train = sample_per_class(train_df, 'class_name', 200, RANDOM_SEED).copy()
    ft_valid = sample_per_class(valid_df, 'class_name', 60, RANDOM_SEED).copy()
    ft_test = sample_per_class(test_df, 'class_name', 60, RANDOM_SEED).copy()

    class TextDataset(torch.utils.data.Dataset):
        def __init__(self, frame, tokenizer, max_length=128):
            self.labels = frame['label'].astype(int).tolist()
            self.encodings = tokenizer(
                frame['text'].tolist(),
                truncation=True,
                max_length=max_length
            )

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            item = {key: torch.tensor(value[idx]) for key, value in self.encodings.items()}
            item['labels'] = torch.tensor(self.labels[idx])
            return item

    train_dataset = TextDataset(ft_train, tokenizer)
    valid_dataset = TextDataset(ft_valid, tokenizer)
    test_dataset = TextDataset(ft_test, tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        id2label={0: 'ham', 1: 'spam'},
        label2id={'ham': 0, 'spam': 1}
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = logits.argmax(axis=-1)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary', zero_division=0)
        acc = accuracy_score(labels, preds)
        return {'accuracy': acc, 'precision': precision, 'recall': recall, 'f1': f1}

    training_args_kwargs = {
        'output_dir': '../transformer_outputs/sms_spam_classifier',
        'learning_rate': 2e-5,
        'per_device_train_batch_size': 8,
        'per_device_eval_batch_size': 16,
        'num_train_epochs': 1,
        'weight_decay': 0.01,
        'save_strategy': 'no',
        'logging_steps': 20,
        'report_to': []
    }
    training_arg_params = inspect.signature(TrainingArguments).parameters
    if 'eval_strategy' in training_arg_params:
        training_args_kwargs['eval_strategy'] = 'epoch'
    else:
        training_args_kwargs['evaluation_strategy'] = 'epoch'

    training_args = TrainingArguments(**training_args_kwargs)

    trainer_kwargs = {
        'model': model,
        'args': training_args,
        'train_dataset': train_dataset,
        'eval_dataset': valid_dataset,
        'data_collator': data_collator,
        'compute_metrics': compute_metrics,
    }
    trainer_params = inspect.signature(Trainer).parameters
    if 'processing_class' in trainer_params:
        trainer_kwargs['processing_class'] = tokenizer
    else:
        trainer_kwargs['tokenizer'] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    start = time.time()
    trainer.train()
    elapsed = time.time() - start
    print(f'fine-tuning elapsed seconds: {elapsed:.1f}')

    predictions = trainer.predict(test_dataset)
    pred_labels = predictions.predictions.argmax(axis=-1)
    true_labels = np.array(ft_test['label'].astype(int))
    target_names = ['ham', 'spam']

    print(classification_report(true_labels, pred_labels, target_names=target_names))
    ConfusionMatrixDisplay.from_predictions(true_labels, pred_labels, display_labels=target_names, cmap='Blues')
    plt.title(f'Fine-tuned {MODEL_NAME} on SMS spam')
    plt.tight_layout()

    fine_tuning_history = pd.DataFrame(trainer.state.log_history)
    display(fine_tuning_history.tail())

    loss_history = fine_tuning_history[[col for col in ['step', 'epoch', 'loss', 'eval_loss'] if col in fine_tuning_history.columns]].copy()
    if {'step', 'loss'}.issubset(loss_history.columns) and loss_history['loss'].notna().any():
        plt.figure(figsize=(6, 4))
        sns.lineplot(data=loss_history.dropna(subset=['loss']), x='step', y='loss', marker='o', label='train loss')
        if 'eval_loss' in loss_history.columns and loss_history['eval_loss'].notna().any():
            sns.scatterplot(data=loss_history.dropna(subset=['eval_loss']), x='step', y='eval_loss', color='#C44E52', s=70, label='validation loss')
        plt.title('Fine-tuning loss history')
        plt.xlabel('Training step')
        plt.ylabel('Loss')
        plt.tight_layout()
else:
    print('Fine-tuning skipped. Set RUN_FINE_TUNING=True to run it.')

## 7. Switching from DistilBERT to BERT or RoBERTa

The training logic stays the same. The model name changes the tokenizer, pretrained weights, and architecture.

Try these values:

```python
MODEL_NAME = 'bert-base-uncased'
MODEL_NAME = 'roberta-base'
```

Practical differences to discuss:

- BERT uses WordPiece; RoBERTa uses byte-level BPE.
- RoBERTa was trained with different pretraining choices than BERT.
- Larger models usually need more memory and time.
- A bigger model is not automatically a better measurement instrument.

In [ ]:
model_comparison_plan = pd.DataFrame([
    {
        'model': 'distilbert-base-uncased',
        'why_use_it': 'fast classroom demo; smaller BERT-style model',
        'main_cost': 'less capacity than BERT/RoBERTa'
    },
    {
        'model': 'bert-base-uncased',
        'why_use_it': 'canonical BERT baseline',
        'main_cost': 'slower and heavier than DistilBERT'
    },
    {
        'model': 'roberta-base',
        'why_use_it': 'strong BERT-family encoder with different tokenizer/pretraining',
        'main_cost': 'heavier; tokenizer behavior differs'
    }
])

model_comparison_plan

## 8. Reporting checklist for transformer classification

A paper or replication package should report:

- Exact model checkpoint, e.g. `bert-base-uncased` or `roberta-base`.
- Tokenizer and maximum sequence length.
- Train/validation/test split construction.
- Number of examples and class balance.
- Hyperparameters: learning rate, batch size, epochs, weight decay.
- Hardware and approximate training time.
- Random seed.
- Validation metrics and error analysis.
- Whether the model was evaluated out of domain.
- Whether labels validly measure the construct.

## 9. Student task

Choose one of these tasks:

1. Run the pretrained sentiment pipeline and explain why it is a different task from SMS spam detection.
2. Fine-tune `distilbert-base-uncased` on the SMS spam subset.
3. Repeat fine-tuning with `bert-base-uncased` or `roberta-base` and compare performance, runtime, and errors.
4. Write a short validation memo: did the transformer improve prediction, measurement validity, both, or neither?